## Installation

### Old

In [ ]:
# Install required dependencies
print("Installing required dependencies...")

!sudo apt-get install -y libusb-1.0-0-dev

# Clone the MuJoCo repo for ALOHA
!git clone https://github.com/google-deepmind/mujoco_menagerie.git

# Clone the Wiki-GRx-Models repo for GR1 and copy the urdf file to temp folder
!git clone https://github.com/FFTAI/Wiki-GRx-Models.git
!cp -r Wiki-GRx-Models/GRX/GR1/gr1t2 gr1_assets

# !pip install --ignore-installed blinker
!pip install -q "z3-solver==4.15.4.0" "genesis-world==0.3.11" "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip uninstall -y numpy numba
!pip install -q "numpy==2.2.0" "numba>=0.61.2"
!pip install -q "transformers<5.0.0" "lerobot[intelrealsense,dynamixel]==0.4.3"

print("Dependencies installed, restarting!")
exit()

Installing required dependencies...
Reading package lists... Done
^C
Cloning into 'mujoco_menagerie'...
^C
Cloning into 'Wiki-GRx-Models'...
^C
cp: cannot stat 'Wiki-GRx-Models/GRX/GR1/gr1t2': No such file or directory
^C


### New

In [1]:
from google.colab import userdata
import os

# 1. Install MuJoCo & IK solver
!pip install -q mujoco mink "qpsolvers[osqp]"
!pip install -q "rerun-sdk[all]==0.28.2" imageio[ffmpeg]
!pip install -q "transformers<5.0.0" "lerobot==0.4.3"

# 2. Authenticate and Clone Private Repo
token = userdata.get("GITHUB_TOKEN")
github_url = f"https://{token}@github.com/vedpatwardhan/le-probe.git"
!git clone $github_url

print("✅ MuJoCo Environment Initialized.")
exit()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.3/124.3 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.2/32.2 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ..

## Helpers

In [ ]:
%%writefile gr1_config.py

import numpy as np
import xml.etree.ElementTree as ET
import os

# -----------------------------------------------------------------------------
# GR1 Robot Configuration
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# JOINT NAMES (Canonical Protocol - Updated for Fourier URDF)
# Model Expects LEFT side first (GR00T-N1.5 Standard)
# -----------------------------------------------------------------------------
L_ARM_NAMES = [
    "left_shoulder_pitch_joint",
    "left_shoulder_roll_joint",
    "left_shoulder_yaw_joint",
    "left_elbow_pitch_joint",
    "left_wrist_yaw_joint",
    "left_wrist_roll_joint",
    "left_wrist_pitch_joint",
]
R_ARM_NAMES = [
    "right_shoulder_pitch_joint",
    "right_shoulder_roll_joint",
    "right_shoulder_yaw_joint",
    "right_elbow_pitch_joint",
    "right_wrist_yaw_joint",
    "right_wrist_roll_joint",
    "right_wrist_pitch_joint",
]
L_HAND_NAMES = [
    "L_thumb_proximal_yaw_joint",
    "L_thumb_proximal_pitch_joint",
    "L_index_proximal_joint",
    "L_middle_proximal_joint",
    "L_ring_proximal_joint",
    "L_pinky_proximal_joint",
]
R_HAND_NAMES = [
    "R_thumb_proximal_yaw_joint",
    "R_thumb_proximal_pitch_joint",
    "R_index_proximal_joint",
    "R_middle_proximal_joint",
    "R_ring_proximal_joint",
    "R_pinky_proximal_joint",
]
HEAD_NAMES = ["head_pitch_joint", "head_roll_joint", "head_yaw_joint"]
WAIST_NAMES = ["waist_yaw_joint", "waist_pitch_joint", "waist_roll_joint"]

# DEFINITIVE WIRE PROTOCOL (ZMQ Communication)
# 32-dim protocol aligned with the model's action head
COMPACT_WIRE_JOINTS = (
    L_ARM_NAMES  # 0-6
    + L_HAND_NAMES  # 7-12
    + HEAD_NAMES  # 13-15
    + R_ARM_NAMES  # 16-22
    + R_HAND_NAMES  # 23-28
    + WAIST_NAMES  # 29-31
)
FROZEN_JOINTS = {
    "head_pitch_joint": 0.0,
    "head_roll_joint": 0.0,
    "head_yaw_joint": 0.0,
}

CAMERA_ATTACH_LINK = "right_hand_pitch_link"

# Central URDF Path
URDF_PATH = "/content/gr1_assets/urdf/gr1t2_fourier_hand_6dof.urdf"
if not os.path.exists(URDF_PATH):
    URDF_PATH = "/Users/vedpatwardhan/Desktop/cortex-os/repos/Wiki-GRx-Models/GRX/GR1/gr1t2/urdf/gr1t2_fourier_hand_6dof.urdf"


# -----------------------------------------------------------------------------
# PHYSICAL BONES (Dynamic Sync 🔗)
# -----------------------------------------------------------------------------
def get_urdf_limits(urdf_path, joint_names):
    """Scans URDF XML for joint limit bounds."""
    if not os.path.exists(urdf_path):
        return np.zeros(len(joint_names), dtype=np.float32), np.zeros(
            len(joint_names), dtype=np.float32
        )

    tree = ET.parse(urdf_path)
    root = tree.getroot()
    joint_db = {
        j.get("name"): (
            float(j.find("limit").get("lower", 0.0)),
            float(j.find("limit").get("upper", 0.0)),
        )
        for j in root.findall("joint")
        if j.find("limit") is not None
    }

    mins = [
        joint_db.get(j, (0.0, 0.0))[0] if j not in FROZEN_JOINTS else FROZEN_JOINTS[j]
        for j in joint_names
    ]
    maxs = [
        joint_db.get(j, (0.0, 0.0))[1] if j not in FROZEN_JOINTS else FROZEN_JOINTS[j]
        for j in joint_names
    ]
    return np.array(mins, dtype=np.float32), np.array(maxs, dtype=np.float32)


JOINT_LIMITS_MIN, JOINT_LIMITS_MAX = get_urdf_limits(URDF_PATH, COMPACT_WIRE_JOINTS)

# -----------------------------------------------------------------------------
# DIAGNOSTIC REPORT 📋
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    print(f"\n--- GR1 Protocol Config (URDF: {os.path.basename(URDF_PATH)}) ---")
    print(f"Camera Mount: {CAMERA_ATTACH_LINK}")
    print(f"{'Idx':<4} {'Joint Name':<40} {'Min':<10} {'Max':<10}")
    print("-" * 70)
    for i, name in enumerate(COMPACT_WIRE_JOINTS):
        print(
            f"{i:<4} {name:<40} {JOINT_LIMITS_MIN[i]:<10.3f} {JOINT_LIMITS_MAX[i]:<10.3f}"
        )
    print("-" * 70)
    print(f"Total DOFs: {len(COMPACT_WIRE_JOINTS)}\n")


Writing gr1_config.py


## Rerun Script

### Old

In [ ]:
%%writefile simulation_gr1.py

import numpy as np
import genesis as gs
import rerun as rr
import zmq
import msgpack
import logging
from scipy.spatial.transform import Rotation as R
import torch
from gr1_config import (
    CAMERA_ATTACH_LINK,
    COMPACT_WIRE_JOINTS,
    JOINT_LIMITS_MIN,
    JOINT_LIMITS_MAX,
    URDF_PATH,
)

# -----------------------------------------------------------------------------
# CONFIG & LOGGING
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------------------------------------------------------
# GR00T CLIENT
# -----------------------------------------------------------------------------
class GR1Client:
    """Minimal ZMQ Client for GR00T Inference."""

    def __init__(self, host="localhost", port=5555):
        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REQ)
        self.socket.setsockopt(zmq.RCVTIMEO, 30000)
        self.socket.connect(f"tcp://{host}:{port}")

    def get_action_chunk(self, obs):
        payload = {
            k: (
                {
                    "__is_np_bytes__": True,
                    "data": v.tobytes(),
                    "shape": v.shape,
                    "dtype": str(v.dtype),
                }
                if isinstance(v, np.ndarray)
                else v
            )
            for k, v in obs.items()
        }
        self.socket.send(msgpack.packb(payload, use_bin_type=True))
        resp = msgpack.unpackb(self.socket.recv(), raw=False)
        return resp.get("action", [])


# -----------------------------------------------------------------------------
# SIMULATION ENGINE
# -----------------------------------------------------------------------------
class GR1Simulation:
    def __init__(self, urdf_path=URDF_PATH):
        gs.init(backend=gs.gpu, precision="32", logging_level="info")
        self._init_genesis(urdf_path)

    def _init_genesis(self, urdf_path):
        # Create Scene
        self.scene = gs.Scene(
            show_viewer=False,
            sim_options=gs.options.SimOptions(dt=0.002, substeps=1),
            vis_options=gs.options.VisOptions(
                lights=[
                    {
                        "type": "directional",
                        "dir": (0, 0, -1),
                        "color": (1.0, 1.0, 1.0),
                        "intensity": 2.0,
                    }
                ]
            ),
            renderer=gs.renderers.Rasterizer(),
        )

        # Add entities
        self.plane = self.scene.add_entity(gs.morphs.Plane())
        self.robot = self.scene.add_entity(
            gs.morphs.URDF(file=urdf_path, pos=(-0.2, 0, 0.95), fixed=True)
        )
        self.table = self.scene.add_entity(
            gs.morphs.Box(pos=(0.45, 0, 0.4), size=(0.4, 0.5, 0.8), fixed=True)
        )
        self.cube = self.scene.add_entity(
            gs.morphs.Box(pos=(0.45, -0.2, 0.82), size=(0.04, 0.04, 0.04), fixed=False),
            surface=gs.surfaces.Default(color=(1, 0, 0)),
        )

        # Add cameras for top, left, right and center views
        self.world_cam_top = self.scene.add_camera(
            res=(224, 224), fov=60, pos=(0.3, 0, 1.8), lookat=(0.3, 0, 0.8)
        )
        self.world_cam_right = self.scene.add_camera(
            res=(224, 224), fov=60, pos=(0.3, -1.5, 1.2), lookat=(0.3, 0, 0.8)
        )
        self.world_cam_left = self.scene.add_camera(
            res=(224, 224), fov=60, pos=(0.3, 1.5, 1.2), lookat=(0.3, 0, 0.8)
        )
        self.world_cam_center = self.scene.add_camera(
            res=(224, 224), fov=60, pos=(1.5, 0.0, 1.2), lookat=(0, 0, 0.8)
        )
        self.world_cam_wrist = self.scene.add_camera(
            res=(224, 224),
            fov=110,
            pos=(0, 0, 0.05),
            lookat=(0, 0, 1.0),
            up=(0, -1, 0),
        )

        # Build the scene
        self.scene.build()

        # Build joint DOF mapping (Must match Simulation.py exactly)
        self.joint_dof_map = []
        for idx, joint_name in enumerate(COMPACT_WIRE_JOINTS):
            joint = self.robot.get_joint(joint_name)
            dof_idx = joint.dofs_idx[0]
            limit_min, limit_max = JOINT_LIMITS_MIN[idx], JOINT_LIMITS_MAX[idx]

            # Finger coupling
            coupled_dofs = []
            if "proximal" in joint_name.lower():
                finger_prefix = joint_name.split("_proximal")[0]
                for other_joint in self.robot.joints:
                    if (
                        other_joint.name
                        and finger_prefix in other_joint.name
                        and other_joint.name != joint_name
                        and "proximal" not in other_joint.name.lower()
                    ):
                        coupled_dofs.append(other_joint.dofs_idx[0])

            self.joint_dof_map.append(
                {
                    "dof_idx": dof_idx,
                    "limits": (limit_min, limit_max),
                    "name": joint_name,
                    "coupled": coupled_dofs,
                }
            )

        # Attach wrist camera after build with a transformation matrix
        pos, lookat, up = (
            np.array([0, 0, 0.05]),
            np.array([0, 0, 1.0]),
            np.array([0, -1, 0]),
        )
        z = (lookat - pos) / np.linalg.norm(lookat - pos)
        x = np.cross(up, z) / np.linalg.norm(np.cross(up, z))
        y = np.cross(z, x)
        offset_T = np.eye(4)
        offset_T[:3, :] = np.column_stack([x, y, z, pos])
        self.world_cam_wrist.attach(
            rigid_link=self.robot.get_link("right_hand_pitch_link"),
            offset_T=offset_T,
        )

        # --- SYNC WITH TELEOP DATA ---
        # 1. Stiff Gains (Matched to teleop precision)
        self.robot.set_dofs_kp(np.full(self.robot.n_dofs, 3500.0))
        self.robot.set_dofs_kv(np.full(self.robot.n_dofs, 150.0))

        # 2. Neutral Start (Set to zeros before first step)
        self.robot.set_dofs_position(np.zeros(self.robot.n_dofs, dtype=np.float32))
        self.scene.step()

    def _normalize_state(self, raw_state):
        """Normalizes raw joint positions to [-1, 1] based on joint limits."""
        # Calculate range safely to avoid division by zero
        joint_range = JOINT_LIMITS_MAX - JOINT_LIMITS_MIN

        # Where range is very small (< 1e-4), treat as fixed/zero to avoid nuclear expansion
        safe_range = np.where(joint_range > 1e-4, joint_range, 1e-4)

        normalized_state = (raw_state - JOINT_LIMITS_MIN) / safe_range * 2.0 - 1.0

        # Hard clip to prevent any uninitialized or buggy values from poisoning the dataset
        return np.clip(normalized_state, -2.0, 2.0)

    def get_state_32(self):
        """Extracts 32-dim normalized state: [L_Arm(7), L_Hand(6), Head(3), R_Arm(7), R_Hand(6), Waist(3)]"""
        # Get current position (32-dim)
        q = self.robot.get_dofs_position().cpu().numpy()

        # Iterate over joints and get current state
        raw_state = []
        for name in COMPACT_WIRE_JOINTS:
            joint = self.robot.get_joint(name)
            raw_state.append(q[joint.dofs_idx_local[0]])
        raw_state = np.array(raw_state, dtype=np.float32)

        return self._normalize_state(raw_state)

    def apply_action_32(self, action_32):
        """Action is normalized [-1, 1], unnormalize to Radians then apply.
        Strict protocol implementation for GR00T-N1.5.
        """
        logging.info(
            f"[ZMQ_ACTION_STATS] min: {float(np.min(action_32)):.3f}, max: {float(np.max(action_32)):.3f}, mean: {float(np.mean(action_32)):.3f}"
        )
        target_q = self.robot.get_dofs_position().cpu().numpy()

        # ONLY update the joints that were actually fine-tuned (Teleop Set)
        # 16-18 (R-Shoulder), 23-28 (R-Hand), 29-30 (Waist) - [Wrist Roll 21 Removed]
        TELEOP_INDICES = [16, 17, 18, 23, 24, 25, 26, 27, 28, 29, 30]

        for idx, mapping in enumerate(self.joint_dof_map):
            if idx not in TELEOP_INDICES:
                # Keep other joints at zero (Homed) to avoid noise
                target_q[mapping["dof_idx"]] = 0.0
                continue

            val = action_32[idx]
            val = np.clip(val, -1.0, 1.0)
            limit_min, limit_max = mapping["limits"]
            target_rad = (val + 1.0) / 2.0 * (limit_max - limit_min) + limit_min
            target_q[mapping["dof_idx"]] = target_rad

            # Apply finger coupling (proximal -> distal)
            for c_idx in mapping["coupled"]:
                target_q[c_idx] = target_rad

        # Apply the final computed 32-dim target position
        self.robot.control_dofs_position(target_q)

    def run(self, instruction="Pick up the red cube"):
        client = GR1Client()
        rr.init("gr1_sim", spawn=False)
        rr.connect_grpc(
            "rerun+http://neonh-103-96-40-120.a.free.pinggy.link:34069/proxy"
        )
        logging.info(f"Starting Multi-Step Inference Task: {instruction}")

        # Outer Loop: Number of inference requests to make (75 * 1.6s = 120 seconds)
        for cycle in range(75):
            logging.info(f"--- Inference Cycle {cycle+1}/75 ---")

            # Perception: Capture observation at the start of the horizon
            rgb_top, _, _, _ = self.world_cam_top.render()
            rgb_left, _, _, _ = self.world_cam_left.render()
            rgb_right, _, _, _ = self.world_cam_right.render()
            rgb_center, _, _, _ = self.world_cam_center.render()
            rgb_wrist, _, _, _ = self.world_cam_wrist.render()

            state_32 = self.get_state_32()
            obs = {
                "world_top": rgb_top[..., :3],
                "world_left": rgb_left[..., :3],
                "world_right": rgb_right[..., :3],
                "world_center": rgb_center[..., :3],
                "world_wrist": rgb_wrist[..., :3],
                "state": state_32,
                "instruction": instruction,
            }

            # Inference: Get 16-action chunk
            logging.info("Requesting Action Chunk...")
            actions = client.get_action_chunk(obs)  # returns list of 16 actions

            # Execution: Inner Loop (Each action is 0.1s / 20 physics steps)
            for action_idx, action in enumerate(actions):
                self.apply_action_32(np.array(action))

                # Step physics for 0.1s (50 steps @ 500Hz)
                for _ in range(50):
                    self.scene.step()

                # Render once per action step for 50Hz visual update
                rgb_top_step, _, _, _ = self.world_cam_top.render()
                rgb_left_step, _, _, _ = self.world_cam_left.render()
                rgb_right_step, _, _, _ = self.world_cam_right.render()
                rgb_center_step, _, _, _ = self.world_cam_center.render()
                rgb_wrist_step, _, _, _ = self.world_cam_wrist.render()

                rr.log("world_top", rr.Image(rgb_top_step[..., :3]))
                rr.log("world_left", rr.Image(rgb_left_step[..., :3]))
                rr.log("world_right", rr.Image(rgb_right_step[..., :3]))
                rr.log("world_center", rr.Image(rgb_center_step[..., :3]))
                rr.log("world_wrist", rr.Image(rgb_wrist_step[..., :3]))

                # Log progress
                total_actions = cycle * len(actions) + action_idx
                if total_actions % 10 == 0:
                    logging.info(f"Executed total actions: {total_actions}")

        logging.info("Task Sequence Complete.")


if __name__ == "__main__":
    sim = GR1Simulation()
    sim.run()


Overwriting simulation_gr1.py


### New

In [ ]:
# Run this in the terminal before running simulation_vla.py
# !export RERUN_URL=rerun+http://xbjgh-103-96-40-121.run.pinggy-free.link:41995/proxy

env: RERUN_URL=rerun+http://wgddl-103-96-40-121.run.pinggy-free.link:36493/proxy


## Model Server

In [ ]:
%%writefile server_gr1.py

import zmq
import msgpack
import torch
import numpy as np
import time
import traceback
import json
import os
import argparse
from transformers import PreTrainedModel
from lerobot.policies.groot.modeling_groot import GrootPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.utils.constants import OBS_IMAGES, OBS_STATE

# -----------------------------------------------------------------------------
# 1. HARDWARE & COMPATIBILITY
# -----------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Force FlashAttention fallback for stability on certain GPUs
_orig_check = PreTrainedModel._check_and_adjust_attn_implementation


def patched_check_attn(self, attn_implementation, is_init_check):
    if attn_implementation == "flash_attention_2":
        return "sdpa"
    return _orig_check(self, attn_implementation, is_init_check)


PreTrainedModel._check_and_adjust_attn_implementation = patched_check_attn


# -----------------------------------------------------------------------------
# 2. UNIFIED MODEL SERVER (Simplified Inference Logic)
# -----------------------------------------------------------------------------
class GR00TInferenceServer:
    """
    Unified Inference Server for GR00T-N1.5.
    Uses the factory pre/post processors to handle SigLIP normalization
    and embodiment-specific joint mapping.
    """

    def __init__(
        self,
        embodiment_tag="gr1",
        port=5555,
        model_repo="nvidia/GR00T-N1.5-3B",
    ):
        print(f"--- Initializing GR00T N1.5 Server (Embodiment: {embodiment_tag}) ---")
        self.port = port
        self.model_repo = model_repo
        self.tokenizer = None

        # Load Policy
        self.policy = GrootPolicy.from_pretrained(self.model_repo)
        self.policy.to(DEVICE)
        self.policy.eval()

        # Setup Pre/Post Processors
        # Setting the embodiment tag ensures the pipeline uses the correct joint limits (ID 24 for gr1)
        self.policy.config.embodiment_tag = embodiment_tag

        print("Creating processors...")
        self.preprocessor, _ = make_pre_post_processors(policy_cfg=self.policy.config)

        # Extract Tokenizer (for Debugging)
        self.tokenizer = None
        for step in self.preprocessor.steps:
            if hasattr(step, "proc") and hasattr(step.proc, "tokenizer"):
                self.tokenizer = step.proc.tokenizer
                break
            elif hasattr(step, "tokenizer"):
                self.tokenizer = step.tokenizer
                break

        print("✅ Model & Preprocessor Ready!")

        print(
            f"--- Policy Config Embodiment: {getattr(self.policy.config, 'embodiment_tag', 'None')} ---"
        )

    def log_diagnostics(self, batch, processed_batch, actions_np, instruction):
        # General serialization function
        def serialize(obj):
            if isinstance(obj, torch.Tensor):
                return obj.cpu().detach().numpy().tolist()
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, dict):
                return {k: serialize(v) for k, v in obj.items()}
            return obj

        # Prepare dict for logging current step
        proc_state_np = processed_batch["observation.state"].cpu().detach().numpy()
        log_entry = {
            "timestamp": time.time(),
            "batch": serialize(batch),
            "processed_metrics": {
                "observation.state": proc_state_np.tolist(),
                "task": instruction,
                "bounds": {
                    "min": float(np.min(proc_state_np)),
                    "max": float(np.max(proc_state_np)),
                    "mean": float(np.mean(proc_state_np)),
                },
                "has_dataset_stats": getattr(self.preprocessor, "stats", None)
                is not None,
            },
            "output": serialize(actions_np),
            "output_stats": {
                "min": float(np.min(actions_np)),
                "max": float(np.max(actions_np)),
                "mean": float(np.mean(actions_np)),
            },
        }

        # Load existing inference history if any
        log_file = "inference_history.json"
        history = []
        if os.path.exists(log_file):
            with open(log_file, "r") as f:
                history = json.load(f)

        # Write updated history
        history.append(log_entry)
        with open(log_file, "w") as f:
            json.dump(history, f, indent=2)

    def run(self, host="0.0.0.0"):
        context = zmq.Context()
        socket = context.socket(zmq.REP)
        socket.bind(f"tcp://{host}:{self.port}")
        print(f"🚀 GR-1 Model Server listening on port {self.port}...")

        while True:
            try:
                message = socket.recv()
                req = msgpack.unpackb(message, raw=False)
                print(
                    f"[{time.strftime('%H:%M:%S')}] 📥 Incoming request: '{req.get('instruction')}'"
                )

                # Unpack Inputs from Simulation Client
                unpack_np = lambda d: (
                    np.frombuffer(d["data"], dtype=d["dtype"]).reshape(d["shape"])
                )
                image_world_top_np = unpack_np(req.get("world_top"))
                image_world_left_np = unpack_np(req.get("world_left"))
                image_world_right_np = unpack_np(req.get("world_right"))
                image_world_center_np = unpack_np(req.get("world_center"))
                image_world_wrist_np = unpack_np(req.get("world_wrist"))
                state_np = unpack_np(req.get("state"))
                instruction = req.get("instruction", "Perform the task.")

                # Transform images ((224, 224, 3) -> (3, 224, 224))
                all_imgs_np = np.stack(
                    [
                        image_world_top_np,
                        image_world_left_np,
                        image_world_right_np,
                        image_world_center_np,
                        image_world_wrist_np,
                    ]
                )
                all_images_t = torch.as_tensor(all_imgs_np, dtype=torch.uint8).permute(
                    0, 3, 1, 2
                )
                (
                    image_world_top_t,
                    image_world_left_t,
                    image_world_right_t,
                    image_world_center_t,
                    image_world_wrist_t,
                ) = all_images_t

                # Create 64-dim state (Rosetta Protocol 🧪)
                state_full = np.zeros(64, dtype=np.float32)
                state_full[0:7] = state_np[0:7]  # Left Arm
                state_full[7:13] = state_np[7:13]  # Left Hand
                state_full[19:22] = state_np[13:16]  # Neck/Head
                state_full[22:29] = state_np[16:23]  # Right Arm
                state_full[29:35] = state_np[23:29]  # Right Hand
                state_full[41:44] = state_np[29:32]  # Waist
                state_t = torch.tensor(state_full, dtype=torch.float32)

                # Prepared batch for preprocessor
                batch = {
                    f"{OBS_IMAGES}.world_top": image_world_top_t,
                    f"{OBS_IMAGES}.world_left": image_world_left_t,
                    f"{OBS_IMAGES}.world_right": image_world_right_t,
                    f"{OBS_IMAGES}.world_center": image_world_center_t,
                    f"{OBS_IMAGES}.world_wrist": image_world_wrist_t,
                    OBS_STATE: state_t,
                    "task": instruction,
                    "embodiment_id": torch.tensor(
                        [24], dtype=torch.long, device=DEVICE
                    ),
                }
                start_time = time.time()

                # Preprocessing for tokenization and move to device (no normalization)
                processed_batch = self.preprocessor(batch)
                for k, v in processed_batch.items():
                    if isinstance(v, torch.Tensor):
                        processed_batch[k] = v.to(DEVICE)

                # Run inference
                print("   [DEBUG] Running Model Inference...")
                with torch.inference_mode():
                    action_chunk = self.policy.predict_action_chunk(processed_batch)

                actions_np = action_chunk[0].cpu().numpy()  # (16, 32)
                inference_time = time.time() - start_time
                print(f"   [DEBUG] Inference Time: {inference_time:.2f} seconds")

                # Diagnostic Data Logging
                self.log_diagnostics(batch, processed_batch, actions_np, instruction)

                # Return canonical 32-dim action (Arms, Hands, Neck, Waist)
                payload = {
                    "action": actions_np.tolist(),
                    "diagnostics": {
                        "instruction": instruction,
                        "inference_time_ms": int((time.time() - start_time) * 1000),
                        "chunk_size": actions_np.shape[0],
                    },
                }
                print(
                    f"   [DEBUG] Sending response back. Chunk Size: {actions_np.shape[0]}"
                )
                socket.send(msgpack.packb(payload, use_bin_type=True))

            except Exception as e:
                import traceback

                print(f"❌ Error in Inference Loop: {e}")
                traceback.print_exc()
                socket.send(msgpack.packb({"error": str(e)}, use_bin_type=True))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="GR00T-N1.5 Inference Server")
    parser.add_argument(
        "--model",
        "-m",
        type=str,
        default="nvidia/GR00T-N1.5-3B",
        help="Hugging Face repo ID or local path to the model.",
    )
    parser.add_argument(
        "--port",
        "-p",
        type=int,
        default=5555,
        help="Port to listen on (default: 5555).",
    )
    parser.add_argument(
        "--tag",
        "-t",
        type=str,
        default="gr1",
        help="Embodiment tag (default: gr1).",
    )
    args = parser.parse_args()

    server = GR00TInferenceServer(
        embodiment_tag=args.tag, port=args.port, model_repo=args.model
    )
    server.run()


Overwriting server_gr1.py


## Run

### Old

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!python server_gr1.py --model nvidia/GR00T-N1.5-3B
# !python server_gr1.py --model vedpatwardhan/GR00T-N1.5-finetuned-pickup

2026-03-29 16:33:49.603878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774802029.624663   36484 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774802029.630790   36484 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774802029.647956   36484 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774802029.647989   36484 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774802029.647992   36484 computation_placer.cc:177] computation placer alr

### New

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [9]:
!mkdir run5
!mkdir run6
!cp -r /content/drive/MyDrive/Masters_Project/gr1_pickup_large/run5/checkpoints/015000/pretrained_model run5
!cp -r /content/drive/MyDrive/Masters_Project/gr1_pickup_compact/run6/checkpoints/015000/pretrained_model run6

In [10]:
# !python le-probe/gr00t_server.py
!python le-probe/gr00t_server.py --weights run5/15000/pretrained_model
# !python le-probe/gr00t_server.py --weights run6/015000/pretrained_model

2026-04-14 10:51:59.778919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776163919.804701   11074 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776163919.813605   11074 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776163919.863810   11074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776163919.863846   11074 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776163919.863850   11074 computation_placer.cc:177] computation placer alr

In [ ]:
!ls /content/drive/MyDrive/Masters_Project/gr1_pickup_large/run4/checkpoints/015000/pretrained_model

config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_groot_action_unpack_unnormalize_v1.safetensors
policy_preprocessor.json
policy_preprocessor_step_2_groot_pack_inputs_v3.safetensors
train_config.json


## Infinite Loop

In [ ]:
!rm -rf inference_history.json

In [ ]:
while True: pass

KeyboardInterrupt: 

In [ ]:
%%writefile sample.py

from lerobot.policies.groot.modeling_groot import GrootPolicy
from lerobot.policies.factory import make_pre_post_processors

# 1. Load the policy configuration
policy = GrootPolicy.from_pretrained("nvidia/GR00T-N1.5-3B")
policy.config.embodiment_tag = "gr1"

# 2. Extract the processors
preprocessor, _ = make_pre_post_processors(policy_cfg=policy.config)

# 3. Find the normalization step for the state
normalization_stats = None
for step in preprocessor.steps:
    if hasattr(step, "stats") and "observation.state" in step.stats:
        normalization_stats = step.stats["observation.state"]
        break

# 4. Print the expected input range / statistics
if normalization_stats:
    mean = normalization_stats.get("mean")
    std = normalization_stats.get("std")
    min_val = normalization_stats.get("min")
    max_val = normalization_stats.get("max")

    print("--- Observation State Normalization Stats (First 15 joints) ---")
    if mean is not None:
        print(f"Mean: {mean[:15].tolist()}")
    if std is not None:
        print(f"Std:  {std[:15].tolist()}")
    if min_val is not None:
        print(f"Min:  {min_val[:15].tolist()}")
    if max_val is not None:
        print(f"Max:  {max_val[:15].tolist()}")
else:
    print("Could not find explicit normalization stats for 'observation.state'.")
    print("Pre-processor steps:", [type(step).__name__ for step in preprocessor.steps])


Overwriting sample.py


In [ ]:
!python sample.py

2026-02-26 15:47:05.201995: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772120825.224564   22526 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772120825.231816   22526 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772120825.249277   22526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772120825.249323   22526 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772120825.249327   22526 computation_placer.cc:177] computation placer alr